In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import RobustScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

## Test dataset: MAASTRO 

In [5]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [6]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [7]:
# need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [8]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [9]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [10]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [11]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# Set y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [12]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [13]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [14]:
# Change the name of a column 'DFS_event' in the clincial_test 
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [15]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

## Feature Selection

### COX PLSR 

In [16]:
# Choose features from the result of Cox PLSR in R
selected_features = [
"shape_Elongation",
"shape_MajorAxisLength",
"hpv_related",
"cavum_oris",
"shape_Sphericity",
"uicc8_III-IV",
"shape_Flatness"
]

In [17]:
X_plsr = X.loc[:, selected_features]
X_new = X_plsr.copy()

In [18]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, selected_features]

# Standardization

In [19]:
# Copy the original X for later 
original_X = X.copy()

In [20]:
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Standardize X_new, the new data with the selected features only 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = RobustScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [21]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[X_new.columns]

In [22]:
X_new

,shape_Elongation,shape_MajorAxisLength,hpv_related,cavum_oris,shape_Sphericity,uicc8_III-IV,shape_Flatness
0,0.600926,42.073251,0.0,0,0.761164,0.0,0.535140
1,0.841579,24.613845,0.0,0,0.697049,0.0,0.367109
2,0.772821,48.030294,0.0,1,0.565792,1.0,0.597785
3,0.847727,25.589900,0.0,0,0.684364,0.0,0.405730
4,0.831483,34.684750,0.0,0,0.503142,0.0,0.442406
...,...,...,...,...,...,...,...
134,0.680294,33.069705,1.0,0,0.742102,0.0,0.523608
135,0.758193,41.043692,1.0,0,0.722918,1.0,0.735524
136,0.770113,36.618802,1.0,0,0.652963,0.0,0.648063
137,0.628897,45.870392,1.0,0,0.724255,1.0,0.492193


In [23]:
X_new_std

,shape_Elongation,shape_MajorAxisLength,hpv_related,cavum_oris,shape_Sphericity,uicc8_III-IV,shape_Flatness
0,-0.485459,0.059912,0.0,0,0.704475,0.0,0.064971
1,0.666232,-0.755402,0.0,0,0.101791,0.0,-0.952446
2,0.337179,0.338093,0.0,1,-1.132016,1.0,0.444285
3,0.695653,-0.709822,0.0,0,-0.017443,0.0,-0.718597
4,0.617917,-0.285114,0.0,0,-1.720923,0.0,-0.496529
...,...,...,...,...,...,...,...
134,-0.105628,-0.360532,1.0,0,0.525285,0.0,-0.004852
135,0.267171,0.011834,1.0,0,0.344965,1.0,1.278289
136,0.324219,-0.194798,1.0,0,-0.312609,0.0,0.748713
137,-0.351598,0.237230,1.0,0,0.357529,1.0,-0.195069


In [24]:
MAASTRO_new 

,shape_Elongation,shape_MajorAxisLength,hpv_related,cavum_oris,shape_Sphericity,uicc8_III-IV,shape_Flatness
0,0.765178,50.002093,1,0,0.668072,0,0.610062
1,0.776540,41.753334,0,0,0.669961,1,0.504616
2,0.697164,44.375483,0,0,0.624081,1,0.478604
3,0.574636,46.115989,0,0,0.577624,1,0.446059
4,0.633419,54.394967,1,0,0.630933,0,0.480378
...,...,...,...,...,...,...,...
94,0.882411,34.218615,0,0,0.671754,1,0.577884
95,0.535802,51.046869,0,0,0.632189,1,0.455642
96,0.716610,50.417953,1,0,0.645548,1,0.631485
97,0.665145,44.901412,1,0,0.727488,0,0.628338


In [25]:
MAASTRO_new_std

,shape_Elongation,shape_MajorAxisLength,hpv_related,cavum_oris,shape_Sphericity,uicc8_III-IV,shape_Flatness
0,0.300600,0.430171,1,0,-0.170593,0,0.518623
1,0.354974,0.044973,0,0,-0.152832,1,-0.119851
2,-0.024893,0.167421,0,0,-0.584101,1,-0.277351
3,-0.611274,0.248699,0,0,-1.020797,1,-0.474407
4,-0.329957,0.635308,1,0,-0.519690,0,-0.266607
...,...,...,...,...,...,...,...
94,0.861639,-0.306881,0,0,-0.135979,1,0.323784
95,-0.797117,0.478960,0,0,-0.507883,1,-0.416383
96,0.068172,0.449591,1,0,-0.382318,1,0.648334
97,-0.178126,0.191981,1,0,0.387916,0,0.629283


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [26]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-17 14:05:01,468] A new study created in memory with name: no-name-69708343-0782-4393-9df0-fd3fdbbd8ff0


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.7276595744680852
Fold 4 C-index: 0.7642585551330798
Fold 5 C-index: 0.7381974248927039
[I 2024-04-17 14:05:13,376] Trial 0 finished with value: 0.7145119462550232 and parameters: {}. Best is trial 0 with value: 0.7145119462550232.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7145119462550232], datetime_start=datetime.datetime(2024, 4, 17, 14, 5, 1, 566717), datetime_complete=datetime.datetime(2024, 4, 17, 14, 5, 13, 361910), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7145119462550232


[I 2024-04-17 14:05:13,388] A new study created in memory with name: no-name-9fccb589-bcd0-4e63-8085-6e54d7d05895


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.23538523110319692
Fold 2 IBS: 0.1899261593875249
Fold 3 IBS: 0.19727300356874158
Fold 4 IBS: 0.19757472980917043
Fold 5 IBS: 0.17989392824267253
[I 2024-04-17 14:05:14,046] Trial 0 finished with value: 0.2000106104222613 and parameters: {}. Best is trial 0 with value: 0.2000106104222613.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2000106104222613], datetime_start=datetime.datetime(2024, 4, 17, 14, 5, 13, 462917), datetime_complete=datetime.datetime(2024, 4, 17, 14, 5, 14, 46517), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2000106104222613


In [27]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [28]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.715
train_ibs:  0.2


#### Test

In [29]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [30]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.568
IBS score: 0.263


In [31]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [32]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [33]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 14:05:14,609] A new study created in memory with name: no-name-b3d2a920-efa7-4436-8e09-c30fe433cfba


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5916334661354582
Fold 2 C-index: 0.7189922480620154
Fold 3 C-index: 0.5595744680851064
Fold 4 C-index: 0.7072243346007605
Fold 5 C-index: 0.6759656652360515
[I 2024-04-17 14:05:15,295] Trial 0 finished with value: 0.6506780364238784 and parameters: {}. Best is trial 0 with value: 0.6506780364238784.


[I 2024-04-17 14:05:15,302] A new study created in memory with name: no-name-5ae43f6c-daaf-46e6-b3cb-ae20103bc26c




* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6506780364238784], datetime_start=datetime.datetime(2024, 4, 17, 14, 5, 14, 728214), datetime_complete=datetime.datetime(2024, 4, 17, 14, 5, 15, 295093), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6506780364238784


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2472470961322622
Fold 2 IBS: 0.23203987476422633
Fold 3 IBS: 0.2289818676351831
Fold 4 IBS: 0.2419747644101403
Fold 5 IBS: 0.22939558698202767
[I 2024-04-17 14:05:16,014] Trial 0 finished with value: 0.23592783798476794 and parameters: {}. Best is trial 0 with value: 0.23592783798476794.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592783798476794], datetime_start=datetime.datetime(2024, 4, 17, 14, 5, 15, 539770), datetime_complete=datetime.datetime(2024, 4, 17, 14, 5, 16, 13947), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592783798476794


In [34]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [35]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.651
train_ibs:  0.236


#### Test

In [36]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [37]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.541


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [38]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [39]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 14:05:16,391] A new study created in memory with name: no-name-537abf14-0b63-4089-871c-110016cff113


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.723404255319149
Fold 4 C-index: 0.7680608365019012


[I 2024-04-17 14:05:17,134] A new study created in memory with name: no-name-bcbe7de1-0da2-4d3d-ba10-6437828905bb


Fold 5 C-index: 0.7381974248927039
[I 2024-04-17 14:05:17,130] Trial 0 finished with value: 0.7168117769460121 and parameters: {}. Best is trial 0 with value: 0.7168117769460121.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7168117769460121], datetime_start=datetime.datetime(2024, 4, 17, 14, 5, 16, 427552), datetime_complete=datetime.datetime(2024, 4, 17, 14, 5, 17, 130159), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7168117769460121


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2341863280747747
Fold 2 IBS: 0.18871473870741187
Fold 3 IBS: 0.19822110991533345
Fold 4 IBS: 0.19650803493993982
Fold 5 IBS: 0.1791141254821675
[I 2024-04-17 14:05:17,879] Trial 0 finished with value: 0.1993488674239255 and parameters: {}. Best is trial 0 with value: 0.1993488674239255.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.1993488674239255], datetime_start=datetime.datetime(2024, 4, 17, 14, 5, 17, 213166), datetime_complete=datetime.datetime(2024, 4, 17, 14, 5, 17, 879201), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.1993488674239255


In [40]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [41]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.717
train_ibs:  0.199


#### Test

In [42]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [43]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.571


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.262


In [44]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [45]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 14:05:18,335] A new study created in memory with name: no-name-5c91361f-05fe-442d-af7e-b0719607ec8f


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.723404255319149
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7381974248927039
[I 2024-04-17 14:05:19,045] Trial 0 finished with value: 0.7168117769460121 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7168117769460121.
Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.723404255319149
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7339055793991416
[I 2024-04-17 14:05:19,707] Trial 1 finished with value: 0.7151565950982957 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7168117769460121.
Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.723404255319149
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7339055793991416
[I 2024-04-17 14:05:20,430] Trial 2 finished with value: 0.7151565950982957 and parameters: {'l1_ratio': 0.22692876841884668}. Be

Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.723404255319149
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7381974248927039
[I 2024-04-17 14:05:37,500] Trial 24 finished with value: 0.7168117769460121 and parameters: {'l1_ratio': 0.7602371709740536}. Best is trial 0 with value: 0.7168117769460121.
Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.723404255319149
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7381974248927039
[I 2024-04-17 14:05:38,182] Trial 25 finished with value: 0.7168117769460121 and parameters: {'l1_ratio': 0.891573855041803}. Best is trial 0 with value: 0.7168117769460121.
Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.723404255319149
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7381974248927039
[I 2024-04-17 14:05:38,813] Trial 26 finished with value: 0.7168117769460121 and parameters: {'l1_ratio': 0.4514399534035586}. Best is trial 0 with value: 0.7168117

Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.723404255319149
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7381974248927039
[I 2024-04-17 14:05:53,123] Trial 48 finished with value: 0.7168117769460121 and parameters: {'l1_ratio': 0.7991933234425521}. Best is trial 0 with value: 0.7168117769460121.
Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.723404255319149
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7381974248927039
[I 2024-04-17 14:05:54,010] Trial 49 finished with value: 0.7168117769460121 and parameters: {'l1_ratio': 0.8520147594086461}. Best is trial 0 with value: 0.7168117769460121.
Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.723404255319149
Fold 4 C-index: 0.7680608365019012
Fold 5 C-index: 0.7381974248927039
[I 2024-04-17 14:05:54,926] Trial 50 finished with value: 0.7168117769460121 and parameters: {'l1_ratio': 0.5059499849722756}. B

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.2341965339295303
Fold 2 IBS: 0.18874267310739173
Fold 3 IBS: 0.19814553245276037
Fold 4 IBS: 0.19651163093536023
Fold 5 IBS: 0.1790498109807477
[I 2024-04-17 14:06:32,415] Trial 0 finished with value: 0.19932923628115806 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.19932923628115806.
Fold 1 IBS: 0.23431758973113054
Fold 2 IBS: 0.1887028958426281
Fold 3 IBS: 0.1975913539905288
Fold 4 IBS: 0.1966259517282194
Fold 5 IBS: 0.17880389731179114
[I 2024-04-17 14:06:33,164] Trial 1 finished with value: 0.1992083377208596 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.1992083377208596.
Fold 1 IBS: 0.23428179958077452
Fold 2 IBS: 0.1887257432511915
Fold 3 IBS: 0.1974781947948211
Fold 4 IBS: 0.19668129265657386
Fold 5 IBS: 0.17873453253521282
[I 2024-04-17 14:06:33,938] Trial 2 finished with value: 0.19918031256371477 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.19918031256371477.

Fold 1 IBS: 0.23439716816647496
Fold 2 IBS: 0.1887340498232072
Fold 3 IBS: 0.2287320438949615
Fold 4 IBS: 0.19671600400020503
Fold 5 IBS: 0.17861429459397207
[I 2024-04-17 14:06:49,888] Trial 25 finished with value: 0.20543871209576414 and parameters: {'l1_ratio': 0.10297700483781348}. Best is trial 22 with value: 0.19912319727139485.
Fold 1 IBS: 0.23422637929906345
Fold 2 IBS: 0.1887402516103256
Fold 3 IBS: 0.19774152953496238
Fold 4 IBS: 0.19663176971967364
Fold 5 IBS: 0.1788381909342659
[I 2024-04-17 14:06:50,581] Trial 26 finished with value: 0.19923562421965818 and parameters: {'l1_ratio': 0.3527001258697712}. Best is trial 22 with value: 0.19912319727139485.
Fold 1 IBS: 0.23423873617668486
Fold 2 IBS: 0.18865757086990564
Fold 3 IBS: 0.19801827639065275
Fold 4 IBS: 0.19655155406440863
Fold 5 IBS: 0.17899842660911816
[I 2024-04-17 14:06:51,274] Trial 27 finished with value: 0.199292912822154 and parameters: {'l1_ratio': 0.58416616744615}. Best is trial 22 with value: 0.199123197271

Fold 1 IBS: 0.23431019495683353
Fold 2 IBS: 0.1887154409415021
Fold 3 IBS: 0.1973445501084278
Fold 4 IBS: 0.1967099583590094
Fold 5 IBS: 0.17868203752508852
[I 2024-04-17 14:07:07,143] Trial 50 finished with value: 0.19915243637817226 and parameters: {'l1_ratio': 0.17526057356525065}. Best is trial 22 with value: 0.19912319727139485.
Fold 1 IBS: 0.2344009139349268
Fold 2 IBS: 0.18875111604946618
Fold 3 IBS: 0.19717673205648287
Fold 4 IBS: 0.1967169364686223
Fold 5 IBS: 0.1786323137671258
[I 2024-04-17 14:07:07,880] Trial 51 finished with value: 0.1991356024553248 and parameters: {'l1_ratio': 0.11717503156407644}. Best is trial 22 with value: 0.19912319727139485.
Fold 1 IBS: 0.23436473328704885
Fold 2 IBS: 0.18871667186997113
Fold 3 IBS: 0.19715488661429642
Fold 4 IBS: 0.19676177597432631
Fold 5 IBS: 0.17862265091554613
[I 2024-04-17 14:07:08,700] Trial 52 finished with value: 0.1991241437322378 and parameters: {'l1_ratio': 0.11135131087864078}. Best is trial 22 with value: 0.1991231972

Fold 1 IBS: 0.23438355614879777
Fold 2 IBS: 0.18872680161678496
Fold 3 IBS: 0.22873349671415405
Fold 4 IBS: 0.1967045299911008
Fold 5 IBS: 0.1786145929779028
[I 2024-04-17 14:07:24,882] Trial 75 finished with value: 0.20543259548974807 and parameters: {'l1_ratio': 0.10106436333699645}. Best is trial 68 with value: 0.1991162513353975.
Fold 1 IBS: 0.23432354536961464
Fold 2 IBS: 0.18878074485662752
Fold 3 IBS: 0.1972323004723012
Fold 4 IBS: 0.19672508185405035
Fold 5 IBS: 0.17865371132600033
[I 2024-04-17 14:07:25,790] Trial 76 finished with value: 0.19914307677571882 and parameters: {'l1_ratio': 0.13577605948707372}. Best is trial 68 with value: 0.1991162513353975.
Fold 1 IBS: 0.245346659320654
Fold 2 IBS: 0.22775355665982508
Fold 3 IBS: 0.2288172865049832
Fold 4 IBS: 0.2388757414772828
Fold 5 IBS: 0.2267019336960901
[I 2024-04-17 14:07:26,107] Trial 77 finished with value: 0.23349903553176704 and parameters: {'l1_ratio': 0.04869294464040792}. Best is trial 68 with value: 0.199116251335

In [46]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [47]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.717
train_ibs:  0.199


#### Test

In [48]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [49]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6964995386793018)

test_cindex : 0.569


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.11288124265625091)

test_ibs:  0.262


In [50]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [51]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-17 14:07:44,894] A new study created in memory with name: no-name-601a56f6-a76f-40f5-92b4-03265b432064


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6613545816733067
Fold 2 C-index: 0.7674418604651163
Fold 3 C-index: 0.6510638297872341
Fold 4 C-index: 0.7547528517110266
Fold 5 C-index: 0.6695278969957081
[I 2024-04-17 14:07:56,467] Trial 0 finished with value: 0.7008282041264783 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7008282041264783.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.7364341085271318
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.7927756653992395
Fold 5 C-index: 0.7167381974248928
[I 2024-04-17 14:08:02,137] Trial 1 finished with value: 0.7097710980423982 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 

Fold 5 C-index: 0.723175965665236
[I 2024-04-17 14:09:10,124] Trial 15 finished with value: 0.720046173250796 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 1, 'n_estimators': 41, 'oob_score': False, 'max_samples': 0.8172106754568015, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.20648604556011108, 'warm_start': False}. Best is trial 13 with value: 0.7204513708045972.
Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.6821705426356589
Fold 3 C-index: 0.6
Fold 4 C-index: 0.7547528517110266
Fold 5 C-index: 0.6673819742489271
[I 2024-04-17 14:09:10,466] Trial 16 finished with value: 0.667554300810756 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 14, 'min_samples_leaf': 3, 'max_depth': 1, 'n_estimators': 4, 'oob_score': False, 'max_samples': 0.8031165510675999, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.37909912926450273, 'warm_start': False}. Best is trial 13 with value: 0.7204513708045972.
Fold 1 C-inde

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:09:38,656] Trial 30 finished with value: 0.5 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 20, 'min_samples_leaf': 20, 'max_depth': 9, 'n_estimators': 170, 'oob_score': True, 'max_samples': 0.760159147741993, 'max_features': None, 'min_weight_fraction_leaf': 0.41069818790111035, 'warm_start': True}. Best is trial 27 with value: 0.777379210577212.
Fold 1 C-index: 0.6852589641434262
Fold 2 C-index: 0.7693798449612403
Fold 3 C-index: 0.8127659574468085
Fold 4 C-index: 0.8003802281368821
Fold 5 C-index: 0.7982832618025751
[I 2024-04-17 14:09:41,416] Trial 31 finished with value: 0.7732136512981864 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 19, 'min_samples_leaf': 18, 'max_depth': 13, 'n_estimators': 235, 'oob_score': True, 'max_samples': 0.7844685228092624, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.05627888575582617, 'warm_start': Tru

...

Fold 5 IBS: 0.18616608441835306
[I 2024-04-17 14:18:59,406] Trial 31 finished with value: 0.19580479754519023 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 4, 'min_samples_leaf': 2, 'max_depth': 4, 'n_estimators': 460, 'oob_score': True, 'max_samples': 0.3742524669147902, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.058072702675878746}. Best is trial 26 with value: 0.19472802701530875.
Fold 1 IBS: 0.21680042114818004
Fold 2 IBS: 0.18633469852302703
Fold 3 IBS: 0.20579918371614245
Fold 4 IBS: 0.1907081836641897
Fold 5 IBS: 0.18824112277850577
[I 2024-04-17 14:19:09,871] Trial 32 finished with value: 0.197576721966009 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 4, 'min_samples_leaf': 4, 'max_depth': 5, 'n_estimators': 457, 'oob_score': True, 'max_samples': 0.3792544247914643, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.08827452661142118}. Best is trial 26 with value: 0.19472802701530875.
Fold 1 IBS: 0.21340885426989026
Fold 2 IBS: 0.192495

Fold 1 IBS: 0.21826580427208442
Fold 2 IBS: 0.18587467681641034
Fold 3 IBS: 0.2052036910380681
Fold 4 IBS: 0.18770622456106834
Fold 5 IBS: 0.18950656244053213
[I 2024-04-17 14:22:11,799] Trial 47 finished with value: 0.19731139182563268 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 9, 'min_samples_leaf': 1, 'max_depth': 11, 'n_estimators': 469, 'oob_score': True, 'max_samples': 0.4674743064217755, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.08523218063639464}. Best is trial 26 with value: 0.19472802701530875.
Fold 1 IBS: 0.24609415388725878
Fold 2 IBS: 0.23220299979144535
Fold 3 IBS: 0.22960252169703754
Fold 4 IBS: 0.24122082372935674
Fold 5 IBS: 0.23052166868873505
[I 2024-04-17 14:22:24,390] Trial 48 finished with value: 0.23592843355876666 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 8, 'min_samples_leaf': 13, 'max_depth': 20, 'n_estimators': 497, 'oob_score': True, 'max_samples': 0.20761893689550356, 'max_features': 'sqrt', 'min_weight_fractio

Fold 1 IBS: 0.21966048398107768
Fold 2 IBS: 0.19129772512794055
Fold 3 IBS: 0.20250622585648823
Fold 4 IBS: 0.18868145334994946
Fold 5 IBS: 0.18722401007552733
[I 2024-04-17 14:25:01,991] Trial 63 finished with value: 0.19787397967819667 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 2, 'n_estimators': 444, 'oob_score': True, 'max_samples': 0.6602193953048239, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.04143492559229172}. Best is trial 26 with value: 0.19472802701530875.
Fold 1 IBS: 0.22441948181751695
Fold 2 IBS: 0.20764396947816882
Fold 3 IBS: 0.21126840742544478
Fold 4 IBS: 0.2137784503905903
Fold 5 IBS: 0.19933277094836385
[I 2024-04-17 14:25:03,768] Trial 64 finished with value: 0.21128861601201693 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 3, 'min_samples_leaf': 1, 'max_depth': 6, 'n_estimators': 55, 'oob_score': True, 'max_samples': 0.4764215998907, 'max_features': 'sqrt', 'min_weight_fraction_leaf':

Fold 5 IBS: 0.19626791200555305
[I 2024-04-17 14:27:11,761] Trial 78 finished with value: 0.20463343559043548 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 14, 'min_samples_leaf': 7, 'max_depth': 2, 'n_estimators': 431, 'oob_score': False, 'max_samples': 0.1776414302372801, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.01245609753689763}. Best is trial 73 with value: 0.19353096336497475.
Fold 1 IBS: 0.22506992941916096
Fold 2 IBS: 0.1956675734053738
Fold 3 IBS: 0.2004354181713437
Fold 4 IBS: 0.1985159039517795
Fold 5 IBS: 0.19425564254565922
[I 2024-04-17 14:27:20,877] Trial 79 finished with value: 0.20278889349866347 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 1, 'n_estimators': 486, 'oob_score': False, 'max_samples': 0.2393243379655301, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.001038000069520234}. Best is trial 73 with value: 0.19353096336497475.
Fold 1 IBS: 0.2189041410044779
Fold 2 IBS: 0.19607

Fold 1 IBS: 0.21169240339518186
Fold 2 IBS: 0.1886341595650372
Fold 3 IBS: 0.202140293031865
Fold 4 IBS: 0.19785454404349312
Fold 5 IBS: 0.18716831705434622
[I 2024-04-17 14:29:17,029] Trial 94 finished with value: 0.19749794341798468 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 5, 'n_estimators': 172, 'oob_score': False, 'max_samples': 0.25112197997211044, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.07196182779958107}. Best is trial 82 with value: 0.19336825659964144.
Fold 1 IBS: 0.22897478326996884
Fold 2 IBS: 0.1770364873003973
Fold 3 IBS: 0.226843018801927
Fold 4 IBS: 0.1897067076457147
Fold 5 IBS: 0.20446826637628404
[I 2024-04-17 14:29:21,362] Trial 95 finished with value: 0.20540585267885839 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 11, 'min_samples_leaf': 2, 'max_depth': 4, 'n_estimators': 214, 'oob_score': False, 'max_samples': 0.2756463981676094, 'max_features': None, 'min_weight_fraction_leaf': 

In [52]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [53]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.841
train_ibs:  0.193


#### Test

In [54]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [55]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=9, max_features='log2', max_leaf_nodes=13,
                     max_samples=0.7347203615160294,
                     min_weight_fraction_leaf=0.009228383305913348,
                     n_estimators=388, random_state=123, warm_start=True)

test_cindex:  0.567


RandomSurvivalForest(max_depth=3, max_features='log2', max_leaf_nodes=16,
                     max_samples=0.19913245689586032, min_samples_leaf=4,
                     min_samples_split=3,
                     min_weight_fraction_leaf=0.021423573926650424,
                     n_estimators=458, random_state=123)

test_ibs:  0.247


In [56]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [57]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [58]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 14:29:42,101] A new study created in memory with name: no-name-48e880ba-dc50-4fd5-9652-461db3823f10


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.8212765957446808
Fold 4 C-index: 0.7737642585551331
Fold 5 C-index: 0.7167381974248928
[I 2024-04-17 14:29:43,855] Trial 0 finished with value: 0.7309095025528539 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7309095025528539.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:29:47,811] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.6957364341085271
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.7699619771863118
Fold 5 C-index: 0.6995708154506438
[I 2024-04-17 14:30:35,768] Trial 15 finished with value: 0.7198013235215133 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.7572934917498981.
Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.6782945736434108
Fold 3 C-index: 0.8063829787234043
Fold 4 C-index: 0.7376425855513308
Fold 5 C-index: 0.6888412017167382
[I 2024-04-17 14:30:38,361] Trial 16 finished with value: 0.7065350567715983 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_feat

Fold 1 C-index: 0.649402390438247
Fold 2 C-index: 0.7596899224806202
Fold 3 C-index: 0.8553191489361702
Fold 4 C-index: 0.8250950570342205
Fold 5 C-index: 0.8154506437768241
[I 2024-04-17 14:31:14,200] Trial 30 finished with value: 0.7809914325332163 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 278, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.05879574792157449}. Best is trial 30 with value: 0.7809914325332163.
Fold 1 C-index: 0.6454183266932271
Fold 2 C-index: 0.7596899224806202
Fold 3 C-index: 0.851063829787234
Fold 4 C-index: 0.8193916349809885
Fold 5 C-index: 0.7896995708154506
[I 2024-04-17 14:31:15,666] Trial 31 finished with value: 0.773052656951504 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 281, 'oob_score': False, 'warm_start': True, 'max_features': Non

...

Fold 1 IBS: 0.22598027269401894
Fold 2 IBS: 0.20325616222593693
Fold 3 IBS: 0.1974550403175071
Fold 4 IBS: 0.2074023934687235
Fold 5 IBS: 0.19292642074350533
[I 2024-04-17 14:36:36,479] Trial 30 finished with value: 0.20540405788993837 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.19882440010334107.
Fold 1 IBS: 0.22967288859416762
Fold 2 IBS: 0.2113244797524532
Fold 3 IBS: 0.20298863335242717
Fold 4 IBS: 0.21680618508524163
Fold 5 IBS: 0.2034856086972431
[I 2024-04-17 14:36:45,409] Trial 31 finished with value: 0.21285555909630655 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 2, 'min_samples_leaf': 3, 'max_depth': 8, 'n_estimators': 458, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.38

Fold 1 IBS: 0.22287647469700594
Fold 2 IBS: 0.1940553114390468
Fold 3 IBS: 0.19175155066892782
Fold 4 IBS: 0.19737115000131933
Fold 5 IBS: 0.18562449372445605
[I 2024-04-17 14:38:39,069] Trial 45 finished with value: 0.1983357961061512 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 144, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.45259524202136425, 'min_weight_fraction_leaf': 0.018417023499892313}. Best is trial 42 with value: 0.19812166533830222.
Fold 1 IBS: 0.22916955270668096
Fold 2 IBS: 0.19778087431618338
Fold 3 IBS: 0.1854433526765588
Fold 4 IBS: 0.20181347277974873
Fold 5 IBS: 0.1880563668012256
[I 2024-04-17 14:38:41,821] Trial 46 finished with value: 0.20045272385607946 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 138, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.54

Fold 1 IBS: 0.23220651508936468
Fold 2 IBS: 0.21369563680773865
Fold 3 IBS: 0.20608638014766714
Fold 4 IBS: 0.22151810446466122
Fold 5 IBS: 0.20573567128046852
[I 2024-04-17 14:39:24,999] Trial 60 finished with value: 0.21584846155798004 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 16, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 58, 'oob_score': False, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.8169268332392909, 'min_weight_fraction_leaf': 0.10367333825975117}. Best is trial 42 with value: 0.19812166533830222.
Fold 1 IBS: 0.2476024556204402
Fold 2 IBS: 0.2003214659562389
Fold 3 IBS: 0.19754714531258735
Fold 4 IBS: 0.20444596709444926
Fold 5 IBS: 0.1958957946158515
[I 2024-04-17 14:39:25,633] Trial 61 finished with value: 0.20916256571991348 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 17, 'n_estimators': 12, 'oob_score': False, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.66

Fold 1 IBS: 0.2294093489012678
Fold 2 IBS: 0.20504726666419198
Fold 3 IBS: 0.20006965758583992
Fold 4 IBS: 0.2050072246769953
Fold 5 IBS: 0.19894671385728707
[I 2024-04-17 14:39:50,097] Trial 75 finished with value: 0.20769604233711642 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 20, 'min_samples_leaf': 4, 'max_depth': 10, 'n_estimators': 60, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.24692331207272655, 'min_weight_fraction_leaf': 0.039366648465568074}. Best is trial 42 with value: 0.19812166533830222.
Fold 1 IBS: 0.22918583286229247
Fold 2 IBS: 0.2105362939816168
Fold 3 IBS: 0.20102170693056107
Fold 4 IBS: 0.2158880397581989
Fold 5 IBS: 0.2037774224979302
[I 2024-04-17 14:39:54,012] Trial 76 finished with value: 0.2120818592061199 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 20, 'min_samples_leaf': 1, 'max_depth': 11, 'n_estimators': 208, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.3

Fold 1 IBS: 0.22333454988172624
Fold 2 IBS: 0.20471686652589108
Fold 3 IBS: 0.1965824831525379
Fold 4 IBS: 0.20553535669394332
Fold 5 IBS: 0.1931394895314824
[I 2024-04-17 14:40:50,735] Trial 90 finished with value: 0.20466174915711619 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 5, 'max_depth': 13, 'n_estimators': 410, 'oob_score': False, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.6447189305878098, 'min_weight_fraction_leaf': 0.07726698990049516}. Best is trial 88 with value: 0.19802180345995218.
Fold 1 IBS: 0.22258797570811703
Fold 2 IBS: 0.19713732800880507
Fold 3 IBS: 0.19548622953950795
Fold 4 IBS: 0.19512550692672126
Fold 5 IBS: 0.18346894725823412
[I 2024-04-17 14:40:58,764] Trial 91 finished with value: 0.19876119748827709 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 20, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 436, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples':

In [59]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [60]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.841
train_ibs:  0.198


#### Test

In [61]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [62]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=14, max_features=0.1, max_leaf_nodes=19,
                   max_samples=0.953229637324892, min_samples_leaf=1,
                   min_samples_split=2,
                   min_weight_fraction_leaf=0.0029509030346900195,
                   n_estimators=77, random_state=123, warm_start=True)

C-index score: 0.581


ExtraSurvivalTrees(max_depth=12, max_leaf_nodes=20,
                   max_samples=0.645870061571142, min_samples_split=5,
                   min_weight_fraction_leaf=0.04343316649769846,
                   n_estimators=432, random_state=123, warm_start=True)

IBS: 0.229


In [63]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis


#### Train

In [64]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-17 14:42:00,920] A new study created in memory with name: no-name-66454e24-0af7-4c34-9226-c817035d4107


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:42:39,458] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:42:59,366] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:53:48,387] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7029978834112648.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:55:08,279] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 15:11:00,574] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.7029978834112648.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 15:11:47,260] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-17 17:04:03,082] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.918589848048704, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.1366452847323028, 'n_estimators': 388, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.18967222528443176, 'max_features': 'auto', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 18}. Best is trial 22 with value: 0.23482614463440274.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-17 17:05:02,766] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7513175598857118, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.35760

Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.2293955930480925
[I 2024-04-17 17:13:32,321] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8952643616873973, 'learning_rate': 0.05359198006915804, 'dropout_rate': 0.6509686174553241, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.713342732410399, 'min_weight_fraction_leaf': 0.037327349410482574, 'max_features': 'auto', 'min_impurity_decrease': 2.2946753237767036e-06, 'validation_fraction': 0.8670144195682054, 'min_samples_split': 9, 'max_leaf_nodes': 20, 'min_samples_leaf': 14, 'max_depth': 18}. Best is trial 22 with value: 0.23482614463440274.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-17 17:14:00,826] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9559398584951578, 'learning_rate': 0.001312025546225645, 'dropout_rate': 0.8046

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-17 17:23:06,218] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9040601608260395, 'learning_rate': 0.008028762030775153, 'dropout_rate': 0.1661055390191164, 'n_estimators': 412, 'criterion': 'squared_error', 'ccp_alpha': 0.5920167940309409, 'min_weight_fraction_leaf': 0.20760648939509768, 'max_features': 1, 'min_impurity_decrease': 1.2858411523836384e-06, 'validation_fraction': 0.7519570151291436, 'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 22 with value: 0.23482614463440274.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-17 17:23:38,609] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9412481314247185, 'learning_rate': 0.003888190949209832, 'dropout_rate': 0.625165508

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-17 17:33:23,204] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8783477148356468, 'learning_rate': 0.019145642246983192, 'dropout_rate': 0.2500967923284591, 'n_estimators': 479, 'criterion': 'squared_error', 'ccp_alpha': 1.2742275973203492, 'min_weight_fraction_leaf': 0.11068414611763369, 'max_features': 'auto', 'min_impurity_decrease': 0.0011701047770450368, 'validation_fraction': 0.9552938036430088, 'min_samples_split': 14, 'max_leaf_nodes': 20, 'min_samples_leaf': 20, 'max_depth': 4}. Best is trial 85 with value: 0.23334097573606488.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-17 17:34:35,382] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9225585795675643, 'learning_rate': 0.0028206049164217276, 'dropout_rate': 0.166

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-17 17:44:14,418] Trial 99 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9084360842440081, 'learning_rate': 0.013060448266216875, 'dropout_rate': 0.25443633448028735, 'n_estimators': 468, 'criterion': 'squared_error', 'ccp_alpha': 0.2223066295281712, 'min_weight_fraction_leaf': 0.25005462596245476, 'max_features': 'auto', 'min_impurity_decrease': 2.1305992007975229e-07, 'validation_fraction': 0.4189919199463679, 'min_samples_split': 15, 'max_leaf_nodes': 12, 'min_samples_leaf': 19, 'max_depth': 1}. Best is trial 85 with value: 0.23334097573606488.


* Best trial for IBS: 
 FrozenTrial(number=85, state=TrialState.COMPLETE, values=[0.23334097573606488], datetime_start=datetime.datetime(2024, 4, 17, 17, 28, 49, 550503), datetime_complete=datetime.datetime(2024, 4, 17, 17, 30, 4, 919538), params={'subsample': 0.9481727373988897, 'learning_rate': 0.020229752590

In [65]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [66]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.749
train_ibs:  0.233


#### Test

In [67]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [68]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.008821356078039954,
                                 criterion='squared_error',
                                 dropout_rate=0.5604101432191428,
                                 learning_rate=0.05159117661647313, max_depth=4,
                                 max_features=0.1, max_leaf_nodes=18,
                                 min_impurity_decrease=9.803482393630746e-05,
                                 min_samples_leaf=7, min_samples_split=8,
                                 min_weight_fraction_leaf=0.26023745967507866,
                                 n_estimators=350, random_state=123,
                                 subsample=0.9197287558651637,
                                 validation_fraction=0.8247616419720114)

C-index score: 0.565


GradientBoostingSurvivalAnalysis(ccp_alpha=0.01138981446692314,
                                 criterion='squared_error',
                                 dropout_rate=0.2479619862207481,
                                 learning_rate=0.02022975259096714, max_depth=8,
                                 max_features='auto', max_leaf_nodes=20,
                                 min_impurity_decrease=9.61920586779085e-07,
                                 min_samples_leaf=18, min_samples_split=14,
                                 min_weight_fraction_leaf=0.026767353450782638,
                                 n_estimators=438, random_state=123,
                                 subsample=0.9481727373988897,
                                 validation_fraction=0.973754058089352)

IBS: 0.229


In [69]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [70]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [71]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 17:44:32,485] A new study created in memory with name: no-name-ffedf783-507a-46e2-8ebf-82753a09e751


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6523605150214592
[I 2024-04-17 17:44:33,564] Trial 0 finished with value: 0.6232514766080982 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6232514766080982.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6523605150214592
[I 2024-04-17 17:44:42,428] Trial 1 finished with value: 0.6232514766080982 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6232514766080982.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 1 C-index: 0.5856573705179283
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6958174904942965
Fold 5 C-index: 0.6824034334763949
[I 2024-04-17 17:45:46,337] Trial 19 finished with value: 0.64767587331303 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 12 with value: 0.6550388490816335.
Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6824034334763949
[I 2024-04-17 17:45:53,955] Trial 20 finished with value: 0.6408386482750206 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 12 with value: 0.6550388490816335.
Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.5617021276595745
F

Fold 5 C-index: 0.6695278969957081
[I 2024-04-17 17:47:15,774] Trial 37 finished with value: 0.6358947216824257 and parameters: {'subsample': 0.22877221290518623, 'dropout_rate': 0.5102437401048863, 'n_estimators': 437, 'learning_rate': 0.059207327336158105}. Best is trial 12 with value: 0.6550388490816335.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6501901140684411
Fold 5 C-index: 0.6695278969957081
[I 2024-04-17 17:47:20,104] Trial 38 finished with value: 0.631313522170144 and parameters: {'subsample': 0.3297325755614151, 'dropout_rate': 0.8867789132968811, 'n_estimators': 392, 'learning_rate': 0.08480908032505553}. Best is trial 12 with value: 0.6550388490816335.
Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6781115879828327
[I 2024-04-17 17:47:21,467] Trial 39 finished with value: 0.6392050853

...

Fold 3 IBS: 0.27495840642227154
Fold 4 IBS: 0.2403770465027076
Fold 5 IBS: 0.22389919673997055
[I 2024-04-17 17:53:17,052] Trial 19 finished with value: 0.24208951118640676 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.24532728078212304
Fold 2 IBS: 0.21338053568843193
Fold 3 IBS: 0.23280583204014674
Fold 4 IBS: 0.23050963160208696
Fold 5 IBS: 0.21229213294998353
[I 2024-04-17 17:53:17,524] Trial 20 finished with value: 0.22686308261255445 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.24480948267779246
Fold 2 IBS: 0.217894217650721
Fold 3 IBS: 0.23043919128917217
Fold 4 IBS: 0.23275995527788665
Fold 5 IBS: 0.215372968667489
[I 2024-04-17 17:53:17,980] Trial 21 finis

Fold 4 IBS: 0.23376225132374828
Fold 5 IBS: 0.21475575962300342
[I 2024-04-17 17:53:41,345] Trial 38 finished with value: 0.23328897274995045 and parameters: {'subsample': 0.605229739689187, 'dropout_rate': 0.13272164755980653, 'n_estimators': 176, 'learning_rate': 0.01283698591663533}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.29108106719697524
Fold 2 IBS: 0.21415810554809628
Fold 3 IBS: 0.2917820507930866
Fold 4 IBS: 0.2588982634047282
Fold 5 IBS: 0.24407574900323586
[I 2024-04-17 17:53:41,891] Trial 39 finished with value: 0.25999904718922445 and parameters: {'subsample': 0.6970162917669863, 'dropout_rate': 0.48841341315505027, 'n_estimators': 59, 'learning_rate': 0.06892741183938003}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.2925659515921918
Fold 2 IBS: 0.2140746731129848
Fold 3 IBS: 0.2919432513336724
Fold 4 IBS: 0.257477121073266
Fold 5 IBS: 0.24657769529804316
[I 2024-04-17 17:53:42,742] Trial 40 finished with value: 0.2605277384820316 

Fold 5 IBS: 0.27450442032616973
[I 2024-04-17 17:53:54,927] Trial 57 finished with value: 0.29085967593170914 and parameters: {'subsample': 0.763955514973389, 'dropout_rate': 0.9964559295946985, 'n_estimators': 282, 'learning_rate': 0.02203371126489022}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.2514876544143989
Fold 2 IBS: 0.20473165445913052
Fold 3 IBS: 0.2442786298594191
Fold 4 IBS: 0.2292932477615486
Fold 5 IBS: 0.2122783863810015
[I 2024-04-17 17:53:55,382] Trial 58 finished with value: 0.2284139145750997 and parameters: {'subsample': 0.8210665501105633, 'dropout_rate': 0.9557982366790785, 'n_estimators': 50, 'learning_rate': 0.03153701184110286}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.2450009034050451
Fold 2 IBS: 0.21574021216390366
Fold 3 IBS: 0.2316126494096215
Fold 4 IBS: 0.2313085066143514
Fold 5 IBS: 0.2135215585555951
[I 2024-04-17 17:53:55,633] Trial 59 finished with value: 0.22743676602970336 and parameters: {'subsample': 0.7

Fold 1 IBS: 0.24975605957417094
Fold 2 IBS: 0.20544669926019324
Fold 3 IBS: 0.24292589881032364
Fold 4 IBS: 0.2283240329592914
Fold 5 IBS: 0.21054396005104126
[I 2024-04-17 17:54:15,450] Trial 77 finished with value: 0.22739933013100408 and parameters: {'subsample': 0.5096986308003991, 'dropout_rate': 0.49881327862855496, 'n_estimators': 145, 'learning_rate': 0.010267257888313383}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.24684927097070503
Fold 2 IBS: 0.2094952280230178
Fold 3 IBS: 0.23652719105524916
Fold 4 IBS: 0.22913951054643386
Fold 5 IBS: 0.2106573202990221
[I 2024-04-17 17:54:15,923] Trial 78 finished with value: 0.2265337041788856 and parameters: {'subsample': 0.5939500427136554, 'dropout_rate': 0.7406477151463077, 'n_estimators': 73, 'learning_rate': 0.015420072941943445}. Best is trial 78 with value: 0.2265337041788856.
Fold 1 IBS: 0.24502409439696052
Fold 2 IBS: 0.22205787317352343
Fold 3 IBS: 0.22926533447627775
Fold 4 IBS: 0.2352795597651623
Fold 5 I

Fold 1 IBS: 0.2471425193657016
Fold 2 IBS: 0.20576111765918625
Fold 3 IBS: 0.2415748835282002
Fold 4 IBS: 0.22741639083301954
Fold 5 IBS: 0.20957521984090532
[I 2024-04-17 17:54:28,980] Trial 96 finished with value: 0.2262940262454026 and parameters: {'subsample': 0.34523349906195283, 'dropout_rate': 0.7739280807288145, 'n_estimators': 132, 'learning_rate': 0.010648288131411126}. Best is trial 85 with value: 0.22596006011310088.
Fold 1 IBS: 0.24385115968166024
Fold 2 IBS: 0.21643646571599923
Fold 3 IBS: 0.23116557489710682
Fold 4 IBS: 0.23167317839356094
Fold 5 IBS: 0.21516282794158326
[I 2024-04-17 17:54:30,289] Trial 97 finished with value: 0.22765784132598207 and parameters: {'subsample': 0.37469859653908655, 'dropout_rate': 0.8528934112321543, 'n_estimators': 187, 'learning_rate': 0.003684889317176295}. Best is trial 85 with value: 0.22596006011310088.
Fold 1 IBS: 0.24621689559308768
Fold 2 IBS: 0.2055572040394727
Fold 3 IBS: 0.24230333094691225
Fold 4 IBS: 0.22648768747110257
Fold

In [72]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [73]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.657
train_ibs:  0.224


#### Test

In [74]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [75]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10144570768327635,
                                              learning_rate=0.09251439643756458,
                                              n_estimators=428,
                                              random_state=123,
                                              subsample=0.10637687141780292)

C-index score: 0.536


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.7782160113329367,
                                              learning_rate=0.005308151321564225,
                                              n_estimators=212,
                                              random_state=123,
                                              subsample=0.17778434918103095)

IBS: 0.233


In [76]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [77]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.841,1.5
ExtraSurvivalTrees,0.841,1.5
GradientBoosting,0.749,3.0
CoxLasso,0.717,4.5
CoxElastic,0.717,4.5
CoxPH,0.715,6.0
ComponentwiseGradientBoosting,0.657,7.0
CoxRidge,0.651,8.0


In [78]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.193,1.0
ExtraSurvivalTrees,0.198,2.0
CoxLasso,0.199,3.5
CoxElastic,0.199,3.5
CoxPH,0.200,5.0
ComponentwiseGradientBoosting,0.224,6.0
GradientBoosting,0.233,7.0
CoxRidge,0.236,8.0


In [79]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ExtraSurvivalTrees,0.581,1.0
CoxLasso,0.571,2.0
CoxElastic,0.569,3.0
CoxPH,0.568,4.0
Randomsurvivalforest,0.567,5.0
GradientBoosting,0.565,6.0
CoxRidge,0.541,7.0
ComponentwiseGradientBoosting,0.536,8.0


In [80]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
CoxRidge,0.229,2.0
ExtraSurvivalTrees,0.229,2.0
GradientBoosting,0.229,2.0
ComponentwiseGradientBoosting,0.233,4.0
Randomsurvivalforest,0.247,5.0
CoxLasso,0.262,6.5
CoxElastic,0.262,6.5
CoxPH,0.263,8.0


In [81]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/dfs/robust/plsr/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_dfs_robust_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [82]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-17
